# 🗑️ Waste Classifier — Improved CNN
### Notebook 04 | Model 2: Improved CNN

Building on the Baseline CNN by adding regularization and augmentation.

**Improvements over Baseline:**
- ✅ BatchNorm after every Conv layer
- ✅ Dropout before fully connected layers
- ✅ Data Augmentation (flip, rotate, color jitter)
- ✅ Same optimizer + scheduler + early stopping from utils.py

**Goal:** Reduce overfitting and improve generalization over the 62% baseline

## 📦 Imports & Setup

In [ ]:
import sys
sys.path.append(".")
from utils import *

import torch
import torch.nn as nn
import torch.optim as optim

set_seed(42)
device = get_device()
print("✅ Setup complete")

## 📁 Load Data

In [ ]:
DATA_DIR    = "../data/processed"
RESULTS_DIR = "../results"
MODELS_DIR  = "../models"

dataloaders, dataset_sizes, class_names = get_dataloaders(
    data_dir=DATA_DIR,
    batch_size=32,
    augment=True,               # ← Augmentation ON
    num_workers=0,
    use_weighted_sampler=True,
)

print(f"Classes     : {class_names}")
print(f"Train size  : {dataset_sizes['train']}")
print(f"Val size    : {dataset_sizes['val']}")
print(f"Test size   : {dataset_sizes['test']}")

## 🏗️ Model Architecture — Improved CNN

Same 3 Conv Block structure as baseline but with BatchNorm and Dropout added.

| Layer Block | Details |
|---|---|
| Conv Block 1 | Conv2d(3→32) + BatchNorm + ReLU + MaxPool |
| Conv Block 2 | Conv2d(32→64) + BatchNorm + ReLU + MaxPool |
| Conv Block 3 | Conv2d(64→128) + BatchNorm + ReLU + MaxPool |
| Conv Block 4 | Conv2d(128→256) + BatchNorm + ReLU + MaxPool |
| FC 1 | Flatten → Dropout(0.5) → 1024 + ReLU |
| FC 2 | Dropout(0.3) → 512 + ReLU |
| FC 3 | 512 → 5 (Softmax) |

In [ ]:
class ImprovedCNN(nn.Module):
    def __init__(self, num_classes=5):
        super(ImprovedCNN, self).__init__()

        # ── Conv Block 1 ──────────────────────────────
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),         # 224 → 112
        )

        # ── Conv Block 2 ──────────────────────────────
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),         # 112 → 56
        )

        # ── Conv Block 3 ──────────────────────────────
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),         # 56 → 28
        )

        # ── Conv Block 4 ──────────────────────────────
        self.conv4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),         # 28 → 14
        )

        # ── Fully Connected ───────────────────────────
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(256 * 14 * 14, 1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.fc(x)
        return x


model = ImprovedCNN(num_classes=5).to(device)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

## ⚙️ Training Configuration

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3, verbose=True
)

print("Criterion : CrossEntropyLoss")
print("Optimizer : Adam (lr=0.001)")
print("Scheduler : ReduceLROnPlateau (factor=0.5, patience=3)")
print("Epochs    : 25")
print("Patience  : 7 (early stopping)")

## 🚀 Training

In [ ]:
history = train_model(
    model=model,
    dataloaders=dataloaders,
    dataset_sizes=dataset_sizes,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    num_epochs=25,
    patience=7,
    model_name="improved_cnn",
    results_dir=RESULTS_DIR,
    models_dir=MODELS_DIR,
)

## 📈 Training Curves

In [ ]:
plot_history(
    history,
    model_name="improved_cnn",
    results_dir=RESULTS_DIR,
)

## 🧪 Evaluation on Test Set

In [ ]:
# Load best checkpoint before evaluating
load_checkpoint(model, f"{MODELS_DIR}/improved_cnn_best.pth", device)

# Test loss & accuracy
test_loss, test_acc = evaluate(model, dataloaders["test"], criterion, device)
print(f"\nTest Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc:.4f} ({test_acc*100:.2f}%)")

## 📊 Classification Report & Confusion Matrix

In [ ]:
# Per-class metrics
report = get_classification_report(model, dataloaders["test"], device, class_names)

# Confusion matrix
cm = plot_confusion_matrix(
    model, dataloaders["test"], device, class_names,
    model_name="improved_cnn",
    results_dir=RESULTS_DIR,
)

## 📉 Ablation Study — Baseline vs Improved

In [ ]:
import json

# Load baseline history
with open(f"{RESULTS_DIR}/baseline_cnn_history.json") as f:
    baseline_history = json.load(f)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Val Loss comparison ────────────────────────────────────────
axes[0].plot(baseline_history["val_loss"], label="Baseline CNN", color="tomato",    linestyle="--")
axes[0].plot(history["val_loss"],          label="Improved CNN", color="steelblue")
axes[0].set_title("Val Loss: Baseline vs Improved")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

# ── Val Accuracy comparison ────────────────────────────────────
axes[1].plot(baseline_history["val_acc"], label="Baseline CNN", color="tomato",    linestyle="--")
axes[1].plot(history["val_acc"],          label="Improved CNN", color="steelblue")
axes[1].set_title("Val Accuracy: Baseline vs Improved")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, 1)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle("Ablation Study — Baseline vs Improved CNN", fontsize=14)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/ablation_baseline_vs_improved.png", dpi=150)
plt.show()
print("✅ Saved → results/ablation_baseline_vs_improved.png")

## 💾 Log Metrics

In [ ]:
log_metrics_to_csv("improved_cnn", {
    "test_accuracy"   : round(test_acc, 4),
    "macro_f1"        : round(report["macro avg"]["f1-score"], 4),
    "macro_precision" : round(report["macro avg"]["precision"], 4),
    "macro_recall"    : round(report["macro avg"]["recall"], 4),
})

## ✅ Improved CNN Summary

| Metric | Baseline CNN | Improved CNN |
|---|---|---|
| Test Accuracy | 62% | TBD |
| Macro F1 | 0.59 | TBD |
| Overfitting | Severe | Reduced |
| Augmentation | None | ✅ |
| BatchNorm | None | ✅ |
| Dropout | None | ✅ |

> Results saved to `results/` and metrics logged to `results/all_models_metrics.csv`  
> Next → `05_transfer_learning.ipynb` — ResNet50 pretrained on ImageNet

## 📂 Verify Model Checkpoint

In [ ]:
import os

checkpoint_name = "improved_cnn_best.pth"  # change per notebook
checkpoint_path = os.path.join(MODELS_DIR, checkpoint_name)

if os.path.exists(checkpoint_path):
    size_mb = os.path.getsize(checkpoint_path) / (1024 * 1024)
    print(f"✅ Checkpoint found: {checkpoint_path}")
    print(f"   Size: {size_mb:.2f} MB")
else:
    print(f"❌ Checkpoint NOT found: {checkpoint_path}")